In [ ]:
import os
import json
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error

import torch

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dropout
from preprocessing import prepare_data

In [109]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

In [110]:
def create_sequences(data, feature_cols, target_col, input_len=24, output_len=24):
    X, y, time_list = [], [], []

    for _, group in data.groupby("building_number"):
        group = group.sort_values("date_time").reset_index(drop=True)

        feature_array = group[feature_cols].values
        target_array = group[target_col].values
        time_array = group["date_time"].values

        for i in range(len(group) - input_len - output_len + 1):
            X.append(feature_array[i:i + input_len])
            y.append(target_array[i + input_len:i + input_len + output_len])
            time_list.append(time_array[i + input_len:i + input_len + output_len])

    return np.array(X), np.array(y), np.array(time_list)

In [112]:
def save_loss_plot(history, save_path: Path) -> None:
    plt.figure(figsize=(8, 5))
    plt.plot(history.history["loss"], label="train_loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.title("Training vs Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

In [113]:
def save_validation_prediction_plot(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    save_path: Path,
    sample_idx: int = 0,
) -> None:
    plt.figure(figsize=(12, 5))
    plt.plot(y_true[sample_idx], label="Actual")
    plt.plot(y_pred[sample_idx], label="Predicted")
    plt.title(f"Validation Forecast Sample {sample_idx}")
    plt.xlabel("Forecast Step")
    plt.ylabel("Power Consumption")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

In [ ]:
def main():
    seed_everything(42)

    base_dir = Path.cwd()
    dataset_dir = base_dir / "dataset"
    artifact_dir = base_dir / "artifacts"
    plot_dir = artifact_dir / "plots"
    train_path = dataset_dir / "train.csv"
    building_info_path = dataset_dir / "building_info.csv"
    artifact_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)

    df = prepare_data(train_path, building_info_path)

    label_encoder = LabelEncoder()
    df["building_type_encoded"] = label_encoder.fit_transform(df["building_type"])

    input_len = 24
    output_len = 12
    target_col = "power_consumption"

    feature_cols = [
        "building_type_encoded",
        "total_area",
        "cooling_area",

        "temperature",
        "humidity",
        "windspeed",

        "hour_sin", "hour_cos",
        "dow_sin", "dow_cos",

        "lag_1",
        "lag_24",

        "diff_1",
        "diff_24",

        "roll_mean_24",
        "roll_std_24",
    ]

    train_parts = []
    valid_parts = []

    for _, group in df.groupby("building_number"):
        group = group.sort_values("date_time").reset_index(drop=True)
        split_idx = int(len(group) * 0.8)
        train_parts.append(group.iloc[:split_idx])
        valid_parts.append(group.iloc[split_idx:])

    train_df = pd.concat(train_parts).reset_index(drop=True)
    valid_df = pd.concat(valid_parts).reset_index(drop=True)

    feature_scaler = MinMaxScaler()
    target_scaler = MinMaxScaler()

    train_df[feature_cols] = feature_scaler.fit_transform(train_df[feature_cols])
    valid_df[feature_cols] = feature_scaler.transform(valid_df[feature_cols])

    train_df[[target_col]] = target_scaler.fit_transform(train_df[[target_col]])
    valid_df[[target_col]] = target_scaler.transform(valid_df[[target_col]])

    X_train, y_train, _ = create_sequences(
        train_df, feature_cols, target_col, input_len, output_len
    )
    X_valid, y_valid, time_valid = create_sequences(
        valid_df, feature_cols, target_col, input_len, output_len
    )

    model = Sequential([
        Input(shape=(input_len, len(feature_cols))),
        LSTM(64, return_sequences=True),
        LSTM(32),
        Dense(32, activation="relu"),
        Dense(output_len),
    ])

    model.compile(optimizer="adam", loss="mse")

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_valid, y_valid),
        epochs=30,
        batch_size=64,
        callbacks=[early_stopping],
        verbose=1,
    )

    y_pred = model.predict(X_valid, verbose=0)

    y_valid_inv = target_scaler.inverse_transform(
        y_valid.reshape(-1, 1)
    ).reshape(y_valid.shape)

    y_pred_inv = target_scaler.inverse_transform(
        y_pred.reshape(-1, 1)
    ).reshape(y_pred.shape)

    rmse = float(np.sqrt(mean_squared_error(y_valid_inv.flatten(), y_pred_inv.flatten())))
    print(f"Integrated Model RMSE: {rmse:.4f}")

    plt.figure(figsize=(8, 5))
    plt.plot(history.history["loss"], label="train_loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.title("Training vs Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(plot_dir / "loss_curve.png")
    plt.close()

    sample_indices = [0, min(50, len(y_valid_inv) - 1), min(100, len(y_valid_inv) - 1)]
    sample_indices = sorted(set(sample_indices))

    for sample_idx in sample_indices:
        plt.figure(figsize=(12, 5))
        plt.plot(time_valid[sample_idx], y_valid_inv[sample_idx], label="Actual")
        plt.plot(time_valid[sample_idx], y_pred_inv[sample_idx], label="Predicted")
        plt.title(f"Validation Forecast Sample {sample_idx}")
        plt.xticks(rotation=45)
        plt.xlabel("Time")
        plt.ylabel("Power Consumption")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(plot_dir / f"validation_prediction_sample{sample_idx}.png")
        plt.close()

    model.save(artifact_dir / "model.keras")

    with open(artifact_dir / "feature_scaler.pkl", "wb") as f:
        pickle.dump(feature_scaler, f)

    with open(artifact_dir / "target_scaler.pkl", "wb") as f:
        pickle.dump(target_scaler, f)

    with open(artifact_dir / "label_encoder.pkl", "wb") as f:
        pickle.dump(label_encoder, f)

    metadata = {
        "feature_cols": feature_cols,
        "target_col": target_col,
        "input_len": input_len,
        "output_len": output_len,
        "rmse": rmse,
        "plots": {
            "loss_curve": "plots/loss_curve.png",
            "validation_samples": [
                f"plots/validation_prediction_sample{idx}.png" for idx in sample_indices
            ]
        }
    }

    with open(artifact_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print("Saved artifacts to artifacts/")
    print("Saved plots to artifacts/plots/")

In [115]:
if __name__ == "__main__":
    main()

Epoch 1/30
2285/2285 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 7.9622e-04 - val_loss: 5.0021e-04
Epoch 2/30
2285/2285 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 3.3339e-04 - val_loss: 3.9381e-04
Epoch 3/30
2285/2285 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 2.7568e-04 - val_loss: 3.4902e-04
Epoch 4/30
2285/2285 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 2.4316e-04 - val_loss: 3.3254e-04
Epoch 5/30
2285/2285 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 2.2476e-04 - val_loss: 2.8357e-04
Epoch 6/30
2285/2285 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 2.0589e-04 - val_loss: 2.6356e-04
Epoch 7/30
2285/2285 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 1.9378e-04 - val_loss: 3.0745e-04
Epoch 8/30
2285/2285 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 1.7983e-04 - val_loss: 2.4243e-04
Epoch 9/30
2285/2285 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 1.8239e-04 - val_loss: 2.3834e-04
Epoch 10/30
2285/2285 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 1.6378e-04 - val_loss: 2.5612e-04
Epoch 11/30
2285/2285 ━━━━━━━